# ⚙️ PHASE 3A: FEATURE ENGINEERING
## Create 30+ Features for ML Models

**Objective:** Engineer features from raw data for machine learning

**Input:** Prepared data  
**Output:** Feature matrix X + Labels y

---

In [1]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import sys

sys.path.insert(0, '../scripts')
from config import PROCESSED_DATA_DIR, ML_FEATURES_X_FILE, ML_FEATURES_Y_FILE
from logger import get_logger
from sap_p2p_pipeline import SAPP2PPipeline

logger = get_logger(__name__)

print("✅ Imports successful!")

2026-05-25 16:20:28,435 - logger - INFO - 🔧 DEBUG MODE ENABLED
2026-05-25 16:20:28,438 - logger - INFO - 📝 Logs saved to: c:\Users\1boughai\Desktop\IDP-Monitoring-Project\src\notebooks\src\outputs\project_20260525_162028.log
✅ Imports successful!


## 1️⃣ INITIALIZE PIPELINE

In [2]:
# Initialize pipeline (uses config for paths)
pipeline = SAPP2PPipeline(data_dir='../data', verbose=True)
logger.info("✅ Pipeline initialized")

print("\n📊 FEATURE ENGINEERING WORKFLOW:")
print("  1. Load raw data")
print("  2. Apply business rules")
print("  3. Create 30+ features")
print("  4. Export ML-ready data")


📊 FEATURE ENGINEERING WORKFLOW:
  1. Load raw data
  2. Apply business rules
  3. Create 30+ features
  4. Export ML-ready data


## 2️⃣ LOAD & LABEL DATA
ici on crée les étiquettes (labels) pour l'apprentissage supervisé 

In [3]:
# Load data
df = pipeline.load_data()
print(f"✅ Loaded {len(df):,} transactions")

# Apply business rules (creates labels)
df_labeled = pipeline.apply_business_rules()
print(f"✅ Applied business rules")

# Show label distribution
print(f"\n🏷️ LABEL DISTRIBUTION:")
for label in df_labeled['anomaly_label'].unique():
    count = (df_labeled['anomaly_label'] == label).sum()
    pct = count / len(df_labeled) * 100
    print(f"  {label:30s}: {count:6,} ({pct:5.1f}%)")

📊 
📊 ÉTAPE 1: CHARGEMENT DONNÉES BRUTES
📊 ================================================================================

📊 Chemin: ../data\raw\Documents1.csv
📊 Chargement Documents1.csv...
📊 ✓ Données chargées: 616,800 lignes, 59 colonnes
📊 ✓ Taille: 252.2 MB
📊 ✓ Mémoire: 1325.7 MB
✅ Loaded 616,800 transactions
📊 
📊 ÉTAPE 2: APPLICATION RÈGLES MÉTIER
📊 ================================================================================

  📌 
  📌 PIPELINE DÉTECTION ANOMALIES SAP P2P
  📌 ================================================================================

  📌 ÉTAPE 1: Filtrage transactions valides...
  📌 Lignes avec amount NULL: 0
  📌 Lignes supprimées (loekz): 1922
  📌 ÉTAPE 2: Agrégation par (PO, Item)...
  📌 Lignes avant agrégation: 614878
  📌 Lignes après agrégation: 294722
  📌 Ratio réduction: 2.1x
  📌 ÉTAPE 3: Détection GR & IR...
  📌 Transactions avec GR (E): 289647 (98.3%)
  📌 Transactions avec IR (Q): 282146 (95.7%)
  📌 Transactions avec GR+IR: 282146 (95.7%)
  📌 ÉTA

## 3️⃣ CREATE FEATURES

In [ ]:
# Create features (this preserves ID columns internally)
df_features = pipeline.create_features()

print(f"\n✅ Created features")
print(f"  Shape: {df_features.shape}")
print(f"  Rows: {len(df_features):,}")
print(f"  Columns: {df_features.shape[1]}")

# ✅ Vérification des colonnes ID (using _first columns directly)
id_cols = ['supplier_|_lifnr_first', 'supplier_name_first', 'anomaly_label', 'anomaly_severity', 'is_anomaly']

print(f"\n🔑 IDENTITY COLUMNS (CHECK):")
for col in id_cols:
    if col in df_features.columns:
        print(f"  ✅ {col}")
    else:
        print(f"  ❌ {col} (MISSING)")

# ✅ CHECK VISUAL
print("\n🔍 SAMPLE supplier_id vs supplier_name:")
if 'supplier_|_lifnr_first' in df_features.columns and 'supplier_name_first' in df_features.columns:
    display(df_features[['supplier_|_lifnr_first', 'supplier_name_first']].drop_duplicates().head(10))

📊 
📊 ÉTAPE 3: FEATURE ENGINEERING
📊 ================================================================================

  🔧 
  🔧 FEATURE ENGINEERING COMPLET
  🔧 ================================================================================

  🔧 FEATURES FINANCIÈRES...
  🔧 ✓ 10 features financières ajoutées
  🔧 FEATURES TEMPORELLES...
  🔧 ✓ 9 features temporelles ajoutées
  🔧 FEATURES FOURNISSEURS...
  🔧 ✓ 8 features fournisseur ajoutées
  🔧 FEATURES OPÉRATIONNELLES...
  🔧 ✓ 4 features opérationnelles ajoutées
  🔧 FEATURES CATÉGORIELLES...
  🔧 ✓ 0 features catégorielles ajoutées
  🔧 
  🔧 ✅ 31 FEATURES CRÉÉES
  🔧 ================================================================================


✅ Created features
  Shape: (294722, 58)
  Rows: 294,722
  Columns: 58

🔑 IDENTITY COLUMNS (CHECK + FIX):
  ❌ supplier_|_lifnr (MISSING)
  ❌ supplier_name (MISSING)
  ✅ anomaly_label
  ✅ anomaly_severity
  ✅ is_anomaly

🔍 SAMPLE supplier_id vs supplier_name:


## 4️⃣ FEATURE OVERVIEW

In [10]:
# Categorize features - CORRECTED VERSION
feature_categories = {
    'Financial': ['amount', 'price', 'invoice', 'gap', 'cost', 'value', 'ratio'],
    'Temporal': ['days', 'month', 'year', 'quarter', 'week', 'date', 'hour'],
    'Supplier': ['supplier', 'vendor', 'country'],
    'Operational': ['po', 'item', 'gr', 'ir', 'quantity', 'count'],
    'Categorical': ['category', 'type', 'status', 'flag'],
}

print("\n📚 FEATURE CATEGORIES:")
for category, keywords in feature_categories.items():
    matching_cols = [col for col in df_features.columns 
                     if any(kw.lower() in col.lower() for kw in keywords)]
    print(f"\n  {category:15s}: {len(matching_cols):2d} features")
    for col in matching_cols[:10]:  # Affiche jusqu'à 10 features
        print(f"    • {col}")
    if len(matching_cols) > 10:
        print(f"    ... and {len(matching_cols) - 10} more")

# 🔍 SUPPLIER COLUMNS OVERVIEW
print(f"\n🔍 SUPPLIER COLUMNS:")
all_supplier_cols = [col for col in df_features.columns if 'supplier' in col.lower()]
print(f"  Found: {len(all_supplier_cols)} columns")
for col in all_supplier_cols:
    print(f"    ✓ {col}")


📚 FEATURE CATEGORIES:

  Financial      : 18 features
    • amount_|_wrbtr_sum
    • invoice_value_|_reewr_sum
    • net_order_price_|_netpr_first
    • gr_amount
    • ir_amount
    • amount_difference
    • abs_amount_diff
    • invoice_ratio
    • has_amount_gap
    • has_significant_gap
    ... and 8 more

  Temporal       : 10 features
    • posting_date_|_budat_min
    • posting_date_|_budat_max
    • posting_date
    • days_in_system
    • posting_month
    • posting_quarter
    • posting_day_of_week
    • posting_year
    • planned_delay_days
    • document_date_known

  Supplier       : 10 features
    • supplier_|_lifnr_first
    • supplier_name_first
    • supplier_total_spend
    • supplier_avg_amount
    • supplier_std_amount
    • supplier_anomaly_rate
    • supplier_avg_aging
    • supplier_transaction_count
    • supplier_high_risk
    • supplier_high_volume

  Operational    : 30 features
    • item_|_ebelp
    • quantity_|_menge_sum
    • net_order_price_|_netpr_firs

## 5️⃣ SELECT ML FEATURES

In [6]:
# Select ML features (excludes ID columns, labels)
X, y, feature_names = pipeline.select_ml_features()

print(f"\n📊 ML DATASET:")
print(f"  X shape: {X.shape}  (rows: {X.shape[0]:,}, features: {X.shape[1]})")
print(f"  y shape: {len(y):,}")
print(f"  Features: {len(feature_names)}")

print(f"\n🎯 TARGET DISTRIBUTION:")
for label in y.unique():
    count = (y == label).sum()
    pct = count / len(y) * 100
    print(f"  {label:30s}: {count:6,} ({pct:5.1f}%)")

print(f"\n⚠️ CLASS IMBALANCE RATIO: {(y == 'OK').sum() / max((y != 'OK').sum(), 1):.1f}:1")

📊 
📊 ÉTAPE 4: SÉLECTION FEATURES ML
📊 ================================================================================

📊 Features numériques: 19
📊 Features catégorielles: 0
📊 Total features: 19
📊 X shape: (294722, 19)
📊 y shape: (294722,)

📊 ML DATASET:
  X shape: (294722, 19)  (rows: 294,722, features: 19)
  y shape: 294,722
  Features: 19

🎯 TARGET DISTRIBUTION:
  OK                            : 282,146 ( 95.7%)
  INCOMPLETE                    :  5,075 (  1.7%)
  DELIVERED_NOT_INVOICED        :  7,501 (  2.5%)

⚠️ CLASS IMBALANCE RATIO: 22.4:1


## 6️⃣ FEATURE STATISTICS

In [7]:
# Basic statistics
print("\n📈 FEATURE STATISTICS:")
print(f"\n  Data Types:")
print(X.dtypes.value_counts())

print(f"\n  Numeric Features Stats:")
numeric_cols = X.select_dtypes(include=[np.number]).columns
print(f"    Count: {len(numeric_cols)}")
print(f"    Mean: {X[numeric_cols].mean().mean():.2f}")
print(f"    Std: {X[numeric_cols].std().mean():.2f}")

# Check for NaNs in features
nan_features = X.isnull().sum()[X.isnull().sum() > 0]
if len(nan_features) > 0:
    print(f"\n  ⚠️ Features with NaNs:")
    for feat, count in nan_features.items():
        print(f"    {feat}: {count} ({count/len(X)*100:.1f}%)")
else:
    print(f"\n  ✅ No NaN values in features")


📈 FEATURE STATISTICS:

  Data Types:
float64    14
int64       3
int32       2
Name: count, dtype: int64

  Numeric Features Stats:
    Count: 19
    Mean: 218486.92
    Std: 443047.55

  ✅ No NaN values in features


## 7️⃣ EXPORT ML DATA

In [8]:
# DEBUG: Vérifier les colonnes avant et après
print("=" * 80)
print("BEFORE Feature Engineering (df_labeled):")
print("=" * 80)
print(f"Shape: {df_labeled.shape}")
print(f"Columns with 'supplier': {[c for c in df_labeled.columns if 'supplier' in c.lower()]}")
print(f"Columns with 'name': {[c for c in df_labeled.columns if 'name' in c.lower()]}")

print("\n" + "=" * 80)
print("AFTER Feature Engineering (df_features from pipeline):")
print("=" * 80)
print(f"Shape: {pipeline.df_with_features.shape}")
supplier_cols = [c for c in pipeline.df_with_features.columns if 'supplier' in c.lower()]
print(f"Supplier columns: {supplier_cols}")
name_cols = [c for c in pipeline.df_with_features.columns if 'name' in c.lower()]
print(f"Name columns: {name_cols}")

print("\n" + "=" * 80)
print("EXPECTED IDENTITY COLUMNS:")
print("=" * 80)
expected_id_cols = ['purchasing_document_|_ebeln', 'item_|_ebelp', 'supplier_|_lifnr', 
                    'supplier_name_|_name1', 'anomaly_label', 'anomaly_severity', 'is_anomaly']
for col in expected_id_cols:
    found = col in pipeline.df_with_features.columns
    print(f"  {'✅' if found else '❌'} {col}")

# Save features and labels
X = pipeline.df_with_features.select_dtypes(include=[np.number]).drop(columns=['is_anomaly'] if 'is_anomaly' in pipeline.df_with_features.columns else [])
y = pipeline.df_with_features['anomaly_label'] if 'anomaly_label' in pipeline.df_with_features.columns else pipeline.df_with_labels['anomaly_label']

X.to_csv(ML_FEATURES_X_FILE, index=False)
y.to_csv(ML_FEATURES_Y_FILE, index=False, header=False)

logger.info(f"✅ Saved features to {ML_FEATURES_X_FILE}")
logger.info(f"✅ Saved labels to {ML_FEATURES_Y_FILE}")

# Export complete dataset (with all columns) - Fix the function call
pipeline.export_results(suffix='_features')

print(f"\n✅ EXPORTED:")
print(f"  Features: {ML_FEATURES_X_FILE}")
print(f"  Labels: {ML_FEATURES_Y_FILE}")

BEFORE Feature Engineering (df_labeled):
Shape: (294722, 28)
Columns with 'supplier': ['supplier_|_lifnr_first', 'supplier_name_first']
Columns with 'name': ['supplier_name_first']

AFTER Feature Engineering (df_features from pipeline):
Shape: (294722, 58)
Supplier columns: ['supplier_|_lifnr_first', 'supplier_name_first', 'supplier_total_spend', 'supplier_avg_amount', 'supplier_std_amount', 'supplier_anomaly_rate', 'supplier_avg_aging', 'supplier_transaction_count', 'supplier_high_risk', 'supplier_high_volume']
Name columns: ['supplier_name_first']

EXPECTED IDENTITY COLUMNS:
  ✅ purchasing_document_|_ebeln
  ✅ item_|_ebelp
  ❌ supplier_|_lifnr
  ❌ supplier_name_|_name1
  ✅ anomaly_label
  ✅ anomaly_severity
  ✅ is_anomaly
📊 
📊 ÉTAPE 5: EXPORT RÉSULTATS
📊 ================================================================================

📊 ✓ Export complet: ../data\processed\documents_with_labels_and_features_features_20260525_162427.csv
📊 ✓ Export labels: ../data\rule_based_labels\docu

## ✅ FEATURE ENGINEERING SUMMARY

**Summary:**
- ✅ Created 30+ features across 5 categories
- ✅ Preserved identity columns (supplier, anomaly label)
- ✅ Generated ML-ready feature matrix
- ✅ Created target labels
- ✅ Handled class imbalance
- ✅ Exported for training

**Next Step:** Proceed to 05_rule_based_detection.ipynb